# Estimating building structural systems from footprints: interactive results

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GeomaticsCaminosUPM/ml_structural_system_paper/blob/main/paper_results.ipynb)

> **Warning.** Until the peer-reviewed version of the paper is published, the footprints without a survey label in `data/` are not the real ones. Their geometry, height and other attributes have been altered, so they look similar to the buildings of the area but do not match them. The surveyed buildings, which make up the training, validation and test sets, are real, so every result of the paper can be reproduced.

This notebook walks through the data and results of the paper *Estimating building structural systems from footprints: transferability of machine learning models across three Latin American cities*:

1. Setup
2. The six surveys, with interactive maps
3. What the model sees
4. Training experiments from scratch (a quick subset, or the full paper)
5. Exploring an experiment: spatial split, scores, confusion matrices and maps of predictions and errors
6. The paper's tables and figures

In Google Colab, choose **Runtime > Change runtime type > GPU** first: XGBoost and CatBoost train on the GPU when one is available.

## 1. Setup

In Colab this clones the repository and installs the pinned environment that trains the models (`.venv`, with [uv](https://docs.astral.sh/uv/)). The maps and tables below only need packages Colab already has. Run locally, start Jupyter from the repository root after `uv sync --group notebook`.

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("reproduce_paper.py").exists():
    if not Path("ml_structural_system_paper").exists():
        !git clone --depth 1 https://github.com/GeomaticsCaminosUPM/ml_structural_system_paper.git
    os.chdir("ml_structural_system_paper")
if IN_COLAB:
    !pip install -q uv
    !uv sync --quiet

ROOT = Path.cwd()
PY = ROOT / ".venv" / "bin" / "python"  # the pinned environment used by reproduce_paper.py
sys.path.insert(0, str(ROOT / "analysis"))
!{PY} -c "from ml_structural_system.gpu import cuda_available; print('GPU available:', cuda_available())"

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display
from ipywidgets import interact

import notebook_utils as nu


def show(fig):
    display(fig)
    plt.close(fig)

## 2. The six surveys

Two surveys per city: a single-district survey and a larger-area one. Survey codes are grouped into four classes: adobe (ADO), reinforced concrete (CR), masonry (M) and light steel and wood (S/W, code S in the data). Labeled buildings per city (Table 1 of the paper):

In [ ]:
nu.class_counts()

Choose a city. Each survey is a map layer you can switch on and off, and hovering a building shows its class and attributes.

In [ ]:
interact(nu.survey_map, city=list(nu.SURVEYS));

## 3. What the model sees

Each building is described by 27 attributes: height, construction era, seismic-code level and roof material, 18 plan-shape indices and 5 descriptors of its contacts with neighboring buildings. None of them observes the structural system directly. How well does each one separate the classes?

In [ ]:
FEATURES = ["height", "year", "convex_hull_irregularity", "inertia_circle_ratio", "EC8_compactness",
            "contact_confinementRatio", "contact_force", "NTC23_setbackRatio", "ASCE7_setbackRatio"]
interact(lambda city, feature: show(nu.feature_by_class(city, feature)), city=list(nu.SURVEYS), feature=FEATURES);

Figures 1 to 3 of the paper only need the data. This draws them with the paper's own code.

In [ ]:
!{PY} reproduce_paper.py figures
for name in ["study_area", "model_input", "workflow"]:
    display(Image(filename=f"figures/{name}.jpg", width=900))

## 4. Train experiments from scratch

Every run starts from scratch: earlier splits and models of the chosen experiments are deleted. Each experiment splits its buildings into spatial blocks, then for each of the five models runs a cross-validated search over hyperparameters and imbalance strategies, tunes the decision rule on the validation set and scores the test set.

The 13 experiment names are in `experiments/main.yaml`. `santo_domingo` is the smallest and the quickest to try. `FULL_PAPER = True` runs every experiment, the learning curve and all tables and figures (several hours, even on a GPU).

In [ ]:
EXPERIMENTS = ["santo_domingo"]
FULL_PAPER = False

In [ ]:
if FULL_PAPER:
    !{PY} reproduce_paper.py
elif EXPERIMENTS:
    !{PY} reproduce_paper.py experiments --experiments {" ".join(EXPERIMENTS)}

## 5. Explore an experiment

Experiments with test predictions:

In [ ]:
finished = nu.finished_experiments()
finished or "None yet: train an experiment in Section 4."

**Spatial split.** Touching buildings form blocks, and each block goes entirely to the training, validation or test set.

In [ ]:
if finished:
    interact(nu.split_map, exp=finished);

**Scores on the test set.** Macro F1 is averaged over the classes present in the test set.

In [ ]:
if finished:
    interact(nu.metrics, exp=finished);

In [ ]:
if finished:
    interact(lambda exp, model: show(nu.plot_confusion(exp, model)), exp=finished, model=nu.MODELS);

**Maps of the test buildings.** Layers: the observed class, the class predicted by the chosen model (wrong predictions outlined in red) and how many of the five models agree with their majority vote. Buildings where the models disagree concentrate the errors.

In [ ]:
if finished:
    interact(nu.prediction_map, exp=finished, model=nu.MODELS);

## 6. The paper's tables and figures

Available after the full run (`FULL_PAPER = True`). The LaTeX versions of Tables 1 to 4 are written to `results/tables/`.

In [ ]:
tables = nu.paper_tables()
if not tables:
    print("Run the full paper first (FULL_PAPER = True in Section 4).")
for title, table in tables.items():
    display(Markdown(f"**{title}**"), table)

In [ ]:
for name in ["confusion_matrices", "learning_curve", "transfer_loss", "feature_importance"]:
    path = Path("figures") / f"{name}.jpg"
    if path.exists():
        display(Image(filename=str(path), width=900))